In [4]:
""import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pcntoolkit as ptk
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import spearmanr, norm

def compute_MSLL(y_true, y_pred, std):
    std[std == 0] = 1e-6
    log_prob = -0.5 * np.log(2 * np.pi * std**2) - ((y_true - y_pred)**2) / (2 * std**2)
    return -np.mean(log_prob)

def compute_HBR_normative_analysis(input_csv, feature_name, output_dir):
    os.makedirs(output_dir, exist_ok=True)

    # === 1. Cargar datos y dividir por sitio ===
    df = pd.read_csv(input_csv)
    np.random.seed(42)
    held_out_site = np.random.choice(df[df['group'] == 'MCI']['SITE'].unique())
    print("Held-out site:", held_out_site)

    train_HC = df[(df['group'] == 'HC') & (df['SITE'] != held_out_site)].copy()
    heldout_HC = df[(df['group'] == 'HC') & (df['SITE'] == held_out_site)].copy()
    heldout_MCI = df[(df['group'] == 'MCI') & (df['SITE'] == held_out_site)].copy()

    train_sites = train_HC['SITE'].unique()
    site_code_map = {site: idx for idx, site in enumerate(train_sites)}
    site_code_map[held_out_site] = len(train_sites)

    train_HC['sitenum'] = train_HC['SITE'].map(site_code_map).astype(int)
    heldout_HC['sitenum'] = site_code_map[held_out_site]
    heldout_MCI['sitenum'] = site_code_map[held_out_site]

    processing_dir = os.path.join(output_dir, "processing")
    os.makedirs(processing_dir, exist_ok=True)

    def save_pkl(data, name):
        with open(os.path.join(processing_dir, name), 'wb') as f:
            pickle.dump(pd.DataFrame(data), f)

    X_train = (train_HC['age'] / 100.0).values.reshape(-1, 1)
    Y_train = train_HC[[feature_name]].values
    B_train = train_HC[['sitenum']].values
    save_pkl(X_train, 'X_train.pkl')
    save_pkl(Y_train, 'Y_train.pkl')
    save_pkl(B_train, 'tr_batch.pkl')

    model_dir = os.path.join(output_dir, "model")
    log_dir = os.path.join(output_dir, "log")
    os.makedirs(model_dir, exist_ok=True)
    os.makedirs(log_dir, exist_ok=True)

    ptk.normative.estimate(
        covfile=os.path.join(processing_dir, 'X_train.pkl'),
        respfile=os.path.join(processing_dir, 'Y_train.pkl'),
        trbefile=os.path.join(processing_dir, 'tr_batch.pkl'),
        alg='hbr', binary=True, savemodel=True,
        output_path=model_dir, outputsuffix="_hbr", log_path=log_dir,
        cvfolds=0
    )

    X_adapt = (heldout_HC['age'] / 100.0).values.reshape(-1, 1)
    Y_adapt = heldout_HC[[feature_name]].values
    B_adapt = heldout_HC[['sitenum']].values
    X_test = (heldout_MCI['age'] / 100.0).values.reshape(-1, 1)
    Y_test = heldout_MCI[[feature_name]].values
    B_test = heldout_MCI[['sitenum']].values

    save_pkl(X_adapt, 'X_adapt.pkl')
    save_pkl(Y_adapt, 'Y_adapt.pkl')
    save_pkl(B_adapt, 'adapt_batch.pkl')
    save_pkl(X_test, 'X_test.pkl')
    save_pkl(Y_test, 'Y_test.pkl')
    save_pkl(B_test, 'test_batch.pkl')

    yhat_test, s2_test, z_scores = ptk.normative.transfer(
        covfile=os.path.join(processing_dir, 'X_adapt.pkl'),
        respfile=os.path.join(processing_dir, 'Y_adapt.pkl'),
        trbefile=os.path.join(processing_dir, 'adapt_batch.pkl'),
        testcov=os.path.join(processing_dir, 'X_test.pkl'),
        testresp=os.path.join(processing_dir, 'Y_test.pkl'),
        tsbefile=os.path.join(processing_dir, 'test_batch.pkl'),
        model_path=model_dir, alg='hbr', binary=True,
        output_path=model_dir, outputsuffix="_transfer", log_path=log_dir
    )

    yhat_adapt, s2_adapt, _ = ptk.normative.transfer(
        covfile=os.path.join(processing_dir, 'X_adapt.pkl'),
        respfile=os.path.join(processing_dir, 'Y_adapt.pkl'),
        trbefile=os.path.join(processing_dir, 'adapt_batch.pkl'),
        testcov=os.path.join(processing_dir, 'X_adapt.pkl'),
        testresp=os.path.join(processing_dir, 'Y_adapt.pkl'),
        tsbefile=os.path.join(processing_dir, 'adapt_batch.pkl'),
        model_path=model_dir, alg='hbr', binary=True,
        output_path=model_dir, outputsuffix="_train_pred", log_path=log_dir
    )

    y_true_train = Y_adapt.flatten()
    y_pred_train = yhat_adapt.flatten()
    std_train = np.sqrt(s2_adapt.flatten())

    y_true_test = Y_test.flatten()
    y_pred_test = yhat_test.flatten()
    std_test = np.sqrt(s2_test.flatten())

    ev_train = 1 - np.var(y_true_train - y_pred_train) / np.var(y_true_train)
    ev_test = 1 - np.var(y_true_test - y_pred_test) / np.var(y_true_test)

    mse_train = mean_squared_error(y_true_train, y_pred_train)
    mse_test = mean_squared_error(y_true_test, y_pred_test)

    msll_train = compute_MSLL(y_true_train, y_pred_train, std_train)
    msll_test = compute_MSLL(y_true_test, y_pred_test, std_test)

    r_train, _ = spearmanr(heldout_HC['age'].values.flatten(), y_pred_train)
    r_test, _ = spearmanr(heldout_MCI['age'].values.flatten(), y_pred_test)

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    # TRAIN
    ax = axes[0]
    ax.plot(heldout_HC['age'], y_pred_train, color='blue', label='Mean')
    ax.fill_between(heldout_HC['age'], y_pred_train - 1.96 * std_train, y_pred_train + 1.96 * std_train,
                    color='blue', alpha=0.2, label='95% CI')
    ax.fill_between(heldout_HC['age'], y_pred_train - 2.58 * std_train, y_pred_train + 2.58 * std_train,
                    color='red', alpha=0.1, label='95% PI')
    ax.scatter(heldout_HC['age'], y_true_train, color='purple', s=10, label='HC-train')
    ax.set_title(f"Train\nEV={ev_train:.2f}, MSE={mse_train:.2f}, MSLL={msll_train:.2f}, r={r_train:.2f}")
    ax.legend()

    # TEST + MCI
    ax = axes[1]
    ages_sorted = np.argsort(heldout_MCI['age'])
    ax.plot(heldout_MCI['age'].values[ages_sorted], y_pred_test[ages_sorted], color='blue', label='Mean')
    for z, label in zip([1.96], ['95%']):
        ax.fill_between(heldout_MCI['age'].values[ages_sorted],
                        y_pred_test[ages_sorted] - z * std_test[ages_sorted],
                        y_pred_test[ages_sorted] + z * std_test[ages_sorted],
                        alpha=0.1, label=label)

    z_vals = {1: -2.33, 5: -1.645, 25: -0.674, 50: 0, 75: 0.674, 95: 1.645, 99: 2.33}
    for p, z in z_vals.items():
        ax.plot(heldout_MCI['age'].values[ages_sorted], y_pred_test[ages_sorted] + z * std_test[ages_sorted],
                '--', linewidth=0.6, label=f'{p}%')
        ax.text(heldout_MCI['age'].values[ages_sorted][-1],
                y_pred_test[ages_sorted][-1] + z * std_test[ages_sorted][-1],
                f'{p}%', fontsize=7, color='blue')

    ax.scatter(heldout_MCI['age'], y_true_test, color='red', s=10, label='MCI')
    ax.set_title(f"Test + MCI\nEV={ev_test:.2f}, MSE={mse_test:.2f}, MSLL={msll_test:.2f}, r={r_test:.2f}")
    ax.legend(fontsize=6)

    # Z-score histogram
    ax = axes[2]
    ax.hist(z_scores, bins=20, color='gray', edgecolor='black', orientation='horizontal')
    ax.axhline(y=0, linestyle='--', color='black')
    ax.set_title("Z-score MCI")

    fig.suptitle(f"HBR Normative Analysis: {feature_name}", fontsize=14)
    fig.tight_layout()
    fig.savefig(os.path.join(output_dir, f"HBR_compare_{feature_name}.png"))
    plt.show()


# === EJEMPLO DE USO ===
input_csv = r"D:\MulticentersEEG\Features_2_normativeModel\gamma_40\24_BEST_EPOCHS\AIF_Babiloni\results_harmonize\Age\neuroharmonize\neuroharmonize_osc_abs_O1_age.csv"
feature = "harm_osc_pw_ab_beta_O1"
save_dir = r"D:/MulticentersEEG/Features_2_normativeModel/gamma_40/24_BEST_EPOCHS/AIF_Babiloni/results_harmonize/Age/HBR_results/osc_abs_O1"

compute_HBR_normative_analysis(input_csv, feature, save_dir)


Held-out site: Seoul
inscaler: None
outscaler: None
Processing data in D:/MulticentersEEG/Features_2_normativeModel/gamma_40/24_BEST_EPOCHS/AIF_Babiloni/results_harmonize/Age/HBR_results/osc_abs_O1\processing\Y_train.pkl
Estimating model  1 of 1


Initializing NUTS using jitter+adapt_diag_grad...
Sequential sampling (1 chains in 1 job)
NUTS: [mu_slope_mu, sigma_slope_mu, offset_slope_mu, mu_intercept_mu, sigma_intercept_mu, offset_intercept_mu, mu_sigma, sigma_sigma, sigma]


Output()

Sampling 1 chain for 500 tune and 1_000 draw iterations (500 + 1_000 draws total) took 12718 seconds.
There were 7 divergences after tuning. Increase `target_accept` or reparameterize.
Only one chain was sampled, this makes it impossible to run some convergence checks
Sampling: [y_like]


Output()

...

Normal


Sampling: [y_like]


Output()

Saving model meta-data...
Evaluating the model ...
Writing outputs ...
No meta-data file is found!
Loading data ...
Using HBR transform...
Transferring model  1 of 1
Error: [Errno 2] No such file or directory: 'D:/MulticentersEEG/Features_2_normativeModel/gamma_40/24_BEST_EPOCHS/AIF_Babiloni/results_harmonize/Age/HBR_results/osc_abs_O1\\model\\NM_0_0_estimate.pkl'


FileNotFoundError: [Errno 2] No such file or directory: 'D:/MulticentersEEG/Features_2_normativeModel/gamma_40/24_BEST_EPOCHS/AIF_Babiloni/results_harmonize/Age/HBR_results/osc_abs_O1\\model\\NM_0_0_estimate.pkl'